# 07 – Pulizia dati: albanesi residenti in Italia per regione
Fonte: Istat, demo.istat.it (RCS) · Residenti al 1° gennaio per sesso · 2015, 2020, 2025

In [4]:
import pandas as pd

# Separatore ";" e codice Istat come testo (così "01" non diventa 1)
df = pd.read_csv("dataraw/Dati_cittadinanza_2025.csv", sep=";", dtype={"Codice Istat": str})
print("Righe e colonne:", df.shape)
df.head()

Righe e colonne: (267367, 10)


,Anno,Codice Istat,Denominazione,Codice stato di cittadinanza,Stato di cittadinanza,Zona,Continente,Maschi,Femmine,Totale
0,2025,001001,Agliè,100,Italia,Unione Europea,Europa,1119,1218,2337
1,2025,001001,Agliè,201,Albania,Europa Centro-Orientale,Europa,0,3,3
2,2025,001001,Agliè,602,Argentina,America Centro-Meridionale,America,2,0,2
3,2025,001001,Agliè,605,Brasile,America Centro-Meridionale,America,6,8,14
4,2025,001001,Agliè,209,Bulgaria,Unione Europea,Europa,0,1,1


In [5]:
albanesi = df[df["Stato di cittadinanza"] == "Albania"]

# Le regioni hanno il codice di 2 cifre (escludo "IT", che è il totale Italia)
regioni = albanesi[(albanesi["Codice Istat"].str.len() == 2) & (albanesi["Codice Istat"] != "IT")]
print("Numero di regioni:", len(regioni))

# Controllo: la somma delle regioni deve essere uguale al totale Italia
print("Somma regioni:", regioni["Totale"].sum())
print("Totale Italia:", albanesi[albanesi["Codice Istat"] == "IT"]["Totale"].values[0])

Numero di regioni: 20
Somma regioni: 414622
Totale Italia: 414622


In [6]:
tabelle = []

for anno in [2015, 2020, 2025]:
    d = pd.read_csv(f"dataraw/Dati_cittadinanza_{anno}.csv", sep=";", dtype={"Codice Istat": str})
    d = d[(d["Stato di cittadinanza"] == "Albania") & (d["Codice Istat"].str.len() == 2) & (d["Codice Istat"] != "IT")]
    tabelle.append(d)
    print(anno, "->", len(d), "regioni,", d["Totale"].sum(), "albanesi")

df = pd.concat(tabelle)

2015 -> 20 regioni, 484482 albanesi
2020 -> 20 regioni, 421591 albanesi
2025 -> 20 regioni, 414622 albanesi


In [7]:
df = df[["Anno", "Codice Istat", "Denominazione", "Maschi", "Femmine", "Totale"]]
df = df.rename(columns={
    "Anno": "anno",
    "Codice Istat": "codice_regione",
    "Denominazione": "regione",
    "Maschi": "maschi",
    "Femmine": "femmine",
    "Totale": "totale"
})

# Accorcio i nomi bilingue, così sono più leggibili nei grafici
df["regione"] = df["regione"].replace({
    "Valle d'Aosta/Vallée d'Aoste": "Valle d'Aosta",
    "Trentino-Alto Adige/Südtirol": "Trentino-Alto Adige"
})

# Percentuale di donne per regione
df["perc_femmine"] = (df["femmine"] / df["totale"] * 100).round(1)
df.head()

,anno,codice_regione,regione,maschi,femmine,totale,perc_femmine
226958,2015,01,Piemonte,23521,22694,46215,49.1
227136,2015,02,Valle d'Aosta,517,476,993,47.9
227255,2015,03,Lombardia,51791,49191,100982,48.7
227451,2015,04,Trentino-Alto Adige,6745,6053,12798,47.3
227614,2015,05,Veneto,20290,19691,39981,49.3


In [8]:
# Le 5 regioni con più albanesi nel 2025
df[df["anno"] == 2025].sort_values("totale", ascending=False).head()

,anno,codice_regione,regione,maschi,femmine,totale,perc_femmine
263275,2025,03,Lombardia,41684,40713,82397,49.4
264127,2025,08,Emilia-Romagna,28964,26611,55575,47.9
264306,2025,09,Toscana,28762,26736,55498,48.2
262973,2025,01,Piemonte,19798,18766,38564,48.7
263625,2025,05,Veneto,16018,15083,31101,48.5


In [9]:
df.to_csv("dataclean/albanesi_italia_regioni.csv", index=False)
print("File salvato!", len(df), "righe")

File salvato! 60 righe
